<a href="https://colab.research.google.com/github/zbovaird/HGNN-using-WAF-Logs-for-Red-Agent/blob/main/analzye_waf_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import argparse
import csv
import html
import json
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from statistics import median


SUSPICIOUS_PATH_PATTERNS = [
    re.compile(pattern, re.IGNORECASE)
    for pattern in [
        r"(^|/)\\.env$",
        r"(^|/)\\.git/config$",
        r"(^|/)actuator(?:/|$)",
        r"(^|/)swagger(?:[-/].*)?$",
        r"(^|/)api[-/]docs(?:/|$)",
        r"(^|/)apidocs(?:/|$)",
        r"(^|/)webjars/swagger-ui/index\\.html$",
        r"(^|/)openapi(?:\\.json)?$",
        r"(^|/)graphql(?:/|$)",
        r"(^|/)\\.claude(?:/|$)",
    ]
]

CLEAN_FIELDNAMES = [
    "event_time",
    "timestamp",
    "client_ip",
    "request_host",
    "webacl_id",
    "http_source_id",
    "http_source_name",
    "http_method",
    "http_version",
    "scheme",
    "uri",
    "query_string",
    "query_present",
    "action",
    "terminating_rule_id",
    "terminating_rule_type",
    "ja3",
    "ja4",
    "user_agent",
    "ua_family",
    "referer",
    "referer_present",
    "cookie_present",
    "accept_language_present",
    "accept_encoding_present",
    "connection_value",
    "header_count",
    "suspicious_path",
]


def parse_timestamp(value: str) -> datetime:
    return datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%f%z")


def jaccard(left: set[str], right: set[str]) -> float:
    if not left and not right:
        return 0.0
    union = left | right
    if not union:
        return 0.0
    return len(left & right) / len(union)


def extract_headers(raw_value: str) -> dict[str, str]:
    if not raw_value:
        return {}
    try:
        payload = json.loads(raw_value)
    except json.JSONDecodeError:
        return {}
    headers = {}
    for header in payload.get("httpRequest", {}).get("headers", []):
        name = (header.get("name") or "").strip().lower()
        value = (header.get("value") or "").strip()
        if name:
            headers[name] = value
    return headers


def user_agent_family(user_agent: str) -> str:
    value = user_agent.lower()
    if "firefox" in value:
        return "firefox"
    if "chrome" in value and "edg" not in value:
        return "chrome"
    if "safari" in value and "chrome" not in value:
        return "safari"
    if "curl" in value:
        return "curl"
    if "python" in value:
        return "python"
    if "go-http-client" in value:
        return "go-http-client"
    if "java" in value:
        return "java"
    if value:
        return "other"
    return "missing"


def suspicious_path(uri: str) -> bool:
    return any(pattern.search(uri) for pattern in SUSPICIOUS_PATH_PATTERNS)


def truthy_int(value: bool) -> int:
    return 1 if value else 0


def as_str(value: object) -> str:
    return value if isinstance(value, str) else ""


def as_float(value: object) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str) and value:
        return float(value)
    return 0.0


def as_int(value: object) -> int:
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(value)
    if isinstance(value, str) and value:
        return int(float(value))
    return 0


def as_str_list(value: object) -> list[str]:
    if isinstance(value, list):
        return [item for item in value if isinstance(item, str)]
    return []


def prepare_clean_dataset(raw_csv_path: Path, cleaned_csv_path: Path) -> tuple[Path, int]:
    row_count = 0
    with raw_csv_path.open(newline="", encoding="utf-8-sig") as source_handle, cleaned_csv_path.open(
        "w", newline="", encoding="utf-8"
    ) as cleaned_handle:
        reader = csv.DictReader(source_handle)
        writer = csv.DictWriter(cleaned_handle, fieldnames=CLEAN_FIELDNAMES)
        writer.writeheader()
        for row in reader:
            headers = extract_headers(row.get("_raw") or "")
            user_agent = headers.get("user-agent", "")
            uri = row.get("httpRequest.uri") or ""
            cleaned_row = {
                "event_time": row.get("_time") or "",
                "timestamp": row.get("timestamp") or "",
                "client_ip": row.get("httpRequest.clientIp") or "",
                "request_host": row.get("httpRequest.host") or "",
                "webacl_id": row.get("webaclId") or "",
                "http_source_id": row.get("httpSourceId") or "",
                "http_source_name": row.get("httpSourceName") or "",
                "http_method": row.get("httpRequest.httpMethod") or "",
                "http_version": row.get("httpRequest.httpVersion") or "",
                "scheme": row.get("httpRequest.scheme") or "",
                "uri": uri,
                "query_string": row.get("httpRequest.args") or "",
                "query_present": truthy_int(bool(row.get("httpRequest.args"))),
                "action": row.get("action") or "",
                "terminating_rule_id": row.get("terminatingRuleId") or "",
                "terminating_rule_type": row.get("terminatingRuleType") or "",
                "ja3": row.get("ja3Fingerprint") or "",
                "ja4": row.get("ja4Fingerprint") or "",
                "user_agent": user_agent,
                "ua_family": user_agent_family(user_agent),
                "referer": headers.get("referer", ""),
                "referer_present": truthy_int(bool(headers.get("referer"))),
                "cookie_present": truthy_int(bool(headers.get("cookie"))),
                "accept_language_present": truthy_int(bool(headers.get("accept-language"))),
                "accept_encoding_present": truthy_int(bool(headers.get("accept-encoding"))),
                "connection_value": headers.get("connection", ""),
                "header_count": len(headers),
                "suspicious_path": truthy_int(suspicious_path(uri)),
            }
            writer.writerow(cleaned_row)
            row_count += 1
    return cleaned_csv_path, row_count


def load_seed_values(seed_file: Path | None) -> set[str]:
    if seed_file is None:
        return set()
    with seed_file.open(encoding="utf-8") as handle:
        return {line.strip() for line in handle if line.strip() and not line.startswith("#")}


@dataclass
class Session:
    session_id: str
    ip: str
    start_time: datetime
    end_time: datetime
    requests: int = 0
    blocked_requests: int = 0
    suspicious_requests: int = 0
    hosts: Counter = field(default_factory=Counter)
    webacls: Counter = field(default_factory=Counter)
    uris: Counter = field(default_factory=Counter)
    rules: Counter = field(default_factory=Counter)
    methods: Counter = field(default_factory=Counter)
    ja3s: Counter = field(default_factory=Counter)
    ja4s: Counter = field(default_factory=Counter)
    user_agents: Counter = field(default_factory=Counter)
    ua_families: Counter = field(default_factory=Counter)
    web_source_names: Counter = field(default_factory=Counter)
    referer_present: int = 0
    cookie_present: int = 0
    accept_language_present: int = 0
    accept_encoding_present: int = 0
    query_present: int = 0
    header_count_total: int = 0
    interarrival_seconds: list[float] = field(default_factory=list)
    previous_time: datetime | None = None

    def add_event(self, row: dict[str, str], event_time: datetime) -> None:
        if event_time < self.start_time:
            self.start_time = event_time
        if event_time > self.end_time:
            self.end_time = event_time
        self.requests += 1
        if row.get("action") == "BLOCK":
            self.blocked_requests += 1

        uri = row.get("uri") or ""
        if row.get("suspicious_path") == "1":
            self.suspicious_requests += 1

        self.hosts[row.get("request_host") or ""] += 1
        self.webacls[row.get("webacl_id") or ""] += 1
        self.uris[uri] += 1
        self.rules[row.get("terminating_rule_id") or ""] += 1
        self.methods[row.get("http_method") or ""] += 1
        self.ja3s[row.get("ja3") or ""] += 1
        self.ja4s[row.get("ja4") or ""] += 1
        self.web_source_names[row.get("http_source_name") or ""] += 1

        user_agent = row.get("user_agent") or ""
        self.user_agents[user_agent] += 1
        self.ua_families[row.get("ua_family") or user_agent_family(user_agent)] += 1
        if row.get("referer_present") == "1":
            self.referer_present += 1
        if row.get("cookie_present") == "1":
            self.cookie_present += 1
        if row.get("accept_language_present") == "1":
            self.accept_language_present += 1
        if row.get("accept_encoding_present") == "1":
            self.accept_encoding_present += 1
        if row.get("query_present") == "1":
            self.query_present += 1
        self.header_count_total += int(row.get("header_count") or 0)

        if self.previous_time is not None:
            delta = abs((event_time - self.previous_time).total_seconds())
            self.interarrival_seconds.append(delta)
        self.previous_time = event_time

    @property
    def duration_seconds(self) -> float:
        return abs((self.end_time - self.start_time).total_seconds())

    def to_feature_row(self, block_signal_weight: float = 10.0) -> dict[str, object]:
        hosts = {host for host in self.hosts if host}
        webacls = {webacl for webacl in self.webacls if webacl}
        uris = {uri for uri in self.uris if uri}
        rules = {rule for rule in self.rules if rule}
        ja4s = {value for value in self.ja4s if value}
        ua_families = {family for family in self.ua_families if family}
        interarrival_median = median(self.interarrival_seconds) if self.interarrival_seconds else None
        requests_per_minute = self.requests / max(self.duration_seconds / 60.0, 1.0)
        suspicious_ratio = self.suspicious_requests / self.requests if self.requests else 0.0
        block_ratio = self.blocked_requests / self.requests if self.requests else 0.0
        query_ratio = self.query_present / self.requests if self.requests else 0.0
        cookie_ratio = self.cookie_present / self.requests if self.requests else 0.0
        referer_ratio = self.referer_present / self.requests if self.requests else 0.0
        accept_language_ratio = self.accept_language_present / self.requests if self.requests else 0.0
        accept_encoding_ratio = self.accept_encoding_present / self.requests if self.requests else 0.0
        avg_header_count = self.header_count_total / self.requests if self.requests else 0.0
        ua_churn = len([ua for ua in self.user_agents if ua]) / self.requests if self.requests else 0.0
        dominant_host = self.hosts.most_common(1)[0][0] if self.hosts else ""
        dominant_webacl = self.webacls.most_common(1)[0][0] if self.webacls else ""
        breadth_score = min(len(hosts) / 25.0, 1.0) * 35.0 + min(len(webacls) / 10.0, 1.0) * 15.0
        probe_score = suspicious_ratio * 30.0 + block_ratio * block_signal_weight
        automation_score = (
            min(requests_per_minute / 60.0, 1.0) * 10.0
            + (1.0 - min(cookie_ratio * 2.0, 1.0)) * 5.0
            + (1.0 - min(referer_ratio * 2.0, 1.0)) * 5.0
            + min(ua_churn * 10.0, 5.0)
        )
        anomaly_score = round(min(breadth_score + probe_score + automation_score, 100.0), 2)
        return {
            "session_id": self.session_id,
            "ip": self.ip,
            "start_time": self.start_time.isoformat(),
            "end_time": self.end_time.isoformat(),
            "duration_seconds": round(self.duration_seconds, 3),
            "requests": self.requests,
            "blocked_requests": self.blocked_requests,
            "suspicious_requests": self.suspicious_requests,
            "requests_per_minute": round(requests_per_minute, 3),
            "median_interarrival_seconds": round(interarrival_median, 3) if interarrival_median is not None else "",
            "unique_hosts": len(hosts),
            "unique_webacls": len(webacls),
            "unique_uris": len(uris),
            "unique_rules": len(rules),
            "unique_ja4": len(ja4s),
            "unique_ua_families": len(ua_families),
            "block_ratio": round(block_ratio, 4),
            "suspicious_ratio": round(suspicious_ratio, 4),
            "query_ratio": round(query_ratio, 4),
            "cookie_ratio": round(cookie_ratio, 4),
            "referer_ratio": round(referer_ratio, 4),
            "accept_language_ratio": round(accept_language_ratio, 4),
            "accept_encoding_ratio": round(accept_encoding_ratio, 4),
            "avg_header_count": round(avg_header_count, 3),
            "ua_churn": round(ua_churn, 4),
            "top_host": dominant_host,
            "top_webacl": dominant_webacl,
            "top_rule": self.rules.most_common(1)[0][0] if self.rules else "",
            "top_ua_family": self.ua_families.most_common(1)[0][0] if self.ua_families else "",
            "anomaly_score": anomaly_score,
            "uri_set": sorted(uris),
            "rule_set": sorted(rules),
            "webacl_set": sorted(webacls),
            "ja4_set": sorted(ja4s),
        }


def build_sessions(csv_path: Path, gap_minutes: int, min_requests: int) -> list[Session]:
    sessions: list[Session] = []
    active: dict[str, Session] = {}
    session_index = Counter()
    gap_seconds = gap_minutes * 60

    with csv_path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            ip = row.get("client_ip") or ""
            timestamp_raw = row.get("event_time") or ""
            if not ip or not timestamp_raw:
                continue
            event_time = parse_timestamp(timestamp_raw)
            session = active.get(ip)
            if session is None or session.previous_time is None:
                session_index[ip] += 1
                session = Session(
                    session_id=f"{ip}-{session_index[ip]}",
                    ip=ip,
                    start_time=event_time,
                    end_time=event_time,
                )
                active[ip] = session
            else:
                gap = abs((event_time - session.previous_time).total_seconds())
                if gap > gap_seconds:
                    if session.requests >= min_requests:
                        sessions.append(session)
                    session_index[ip] += 1
                    session = Session(
                        session_id=f"{ip}-{session_index[ip]}",
                        ip=ip,
                        start_time=event_time,
                        end_time=event_time,
                    )
                    active[ip] = session
            session.add_event(row, event_time)

    for session in active.values():
        if session.requests >= min_requests:
            sessions.append(session)
    return sessions


def build_similarity_graph(
    feature_rows: list[dict[str, object]],
    similarity_threshold: float,
    partition_by: str,
) -> tuple[list[dict[str, object]], dict[str, int]]:
    edges: list[dict[str, object]] = []
    adjacency: dict[str, set[str]] = defaultdict(set)

    for index, left in enumerate(feature_rows):
        left_session_id = as_str(left["session_id"])
        left_uris = set(as_str_list(left["uri_set"]))
        left_rules = set(as_str_list(left["rule_set"]))
        left_webacls = set(as_str_list(left["webacl_set"]))
        left_ja4s = set(as_str_list(left["ja4_set"]))
        for right in feature_rows[index + 1:]:
            right_session_id = as_str(right["session_id"])
            if partition_by == "host" and as_str(left["top_host"]) != as_str(right["top_host"]):
                continue
            if partition_by == "webacl" and as_str(left["top_webacl"]) != as_str(right["top_webacl"]):
                continue
            uri_similarity = jaccard(left_uris, set(as_str_list(right["uri_set"])))
            rule_similarity = jaccard(left_rules, set(as_str_list(right["rule_set"])))
            webacl_similarity = jaccard(left_webacls, set(as_str_list(right["webacl_set"])))
            ja4_similarity = jaccard(left_ja4s, set(as_str_list(right["ja4_set"])))
            weighted_similarity = (
                0.45 * uri_similarity
                + 0.20 * rule_similarity
                + 0.20 * webacl_similarity
                + 0.15 * ja4_similarity
            )
            if weighted_similarity < similarity_threshold:
                continue
            edge = {
                "source_session_id": left_session_id,
                "target_session_id": right_session_id,
                "similarity": round(weighted_similarity, 4),
                "uri_similarity": round(uri_similarity, 4),
                "rule_similarity": round(rule_similarity, 4),
                "webacl_similarity": round(webacl_similarity, 4),
                "ja4_similarity": round(ja4_similarity, 4),
            }
            edges.append(edge)
            adjacency[left_session_id].add(right_session_id)
            adjacency[right_session_id].add(left_session_id)

    cluster_id_by_session: dict[str, int] = {}
    next_cluster_id = 1
    for row in feature_rows:
        session_id = as_str(row["session_id"])
        if session_id in cluster_id_by_session:
            continue
        stack = [session_id]
        cluster_id_by_session[session_id] = next_cluster_id
        while stack:
            current = stack.pop()
            for neighbor in adjacency.get(current, set()):
                if neighbor in cluster_id_by_session:
                    continue
                cluster_id_by_session[neighbor] = next_cluster_id
                stack.append(neighbor)
        next_cluster_id += 1
    return edges, cluster_id_by_session


def write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({name: row.get(name, "") for name in fieldnames})


def write_text(path: Path, content: str) -> None:
    path.write_text(content, encoding="utf-8")


def format_html_value(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return html.escape(str(value)) if value is not None else ""


def render_html_table(rows: list[dict[str, object]], columns: list[str], title: str) -> str:
    if not rows:
        return f"<section><h2>{html.escape(title)}</h2><p>No data available.</p></section>"

    header_cells = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{format_html_value(row.get(column, ''))}</td>" for column in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    body = "".join(body_rows)
    return (
        f"<section><h2>{html.escape(title)}</h2>"
        f"<div class=\"table-wrap\"><table><thead><tr>{header_cells}</tr></thead><tbody>{body}</tbody></table></div>"
        "</section>"
    )


def build_html_report(
    cleaned_rows: int,
    feature_rows: list[dict[str, object]],
    edges: list[dict[str, object]],
    cluster_summaries: list[dict[str, object]],
    ip_summaries: list[dict[str, object]],
) -> str:
    summary_cards = [
    ("Cleaned events", cleaned_rows),
    ("Sessions", len(feature_rows)),
    ("Similarity edges", len(edges)),
    ("Clusters", len(cluster_summaries)),
    ("Flagged IPs", len(ip_summaries)),
    ]
    summary_html = "".join(
    f"<article class=\"metric\"><h2>{html.escape(label)}</h2><p>{value}</p></article>"
    for label, value in summary_cards
    )

    top_sessions = sorted(
    feature_rows,
    key=lambda row: (-as_float(row["anomaly_score"]), -as_int(row["requests"])),
    )[:15]
    top_session_rows = [
    {
        "session_id": as_str(row["session_id"]),
        "ip": as_str(row["ip"]),
        "cluster_id": as_int(row["cluster_id"]),
        "score": round(as_float(row["anomaly_score"]), 2),
        "requests": as_int(row["requests"]),
        "hosts": as_int(row["unique_hosts"]),
        "webacls": as_int(row["unique_webacls"]),
        "suspicious_ratio": round(as_float(row["suspicious_ratio"]), 4),
        "block_ratio": round(as_float(row["block_ratio"]), 4),
    }
    for row in top_sessions
    ]

    top_cluster_rows = [
    {
        "cluster_id": as_int(row["cluster_id"]),
        "label": as_str(row["cluster_label"]),
        "avg_score": round(as_float(row["avg_anomaly_score"]), 2),
        "max_score": round(as_float(row["max_anomaly_score"]), 2),
        "sessions": as_int(row["session_count"]),
        "top_host": as_str(row["top_host"]),
        "top_webacl": as_str(row["top_webacl"]),
        "top_ip": as_str(row["top_ip"]),
    }
    for row in cluster_summaries[:15]
    ]

    top_ip_rows = [
    {
        "ip": as_str(row["ip"]),
        "candidate_score": round(as_float(row["candidate_ip_score"]), 2),
        "label": as_str(row["candidate_label"]),
        "sessions": as_int(row["session_count"]),
        "clusters": as_int(row["cluster_count"]),
        "high_confidence_sessions": as_int(row["high_confidence_session_count"]),
        "total_requests": as_int(row["total_requests"]),
        "top_host": as_str(row["top_host"]),
        "top_webacl": as_str(row["top_webacl"]),
    }
    for row in ip_summaries[:15]
    ]

    generated_at = datetime.now().astimezone().isoformat(timespec="seconds")
    return f"""<!doctype html>
<html lang=\"en\">
<head>
    <meta charset=\"utf-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
    <title>WAF Graph Analysis</title>
    <style>
        :root {{
            color-scheme: light;
            --bg: #f3f0ea;
            --panel: #fffdf8;
            --ink: #1f2328;
            --muted: #625b52;
            --line: #d9d0c4;
            --accent: #a64b2a;
            --accent-soft: #f3d9cf;
        }}
        * {{ box-sizing: border-box; }}
        body {{
            margin: 0;
            font-family: Georgia, "Iowan Old Style", "Palatino Linotype", serif;
            background: linear-gradient(180deg, #efe7da 0%, var(--bg) 100%);
            color: var(--ink);
        }}
        main {{ max-width: 1440px; margin: 0 auto; padding: 32px 20px 48px; }}
        header {{ margin-bottom: 24px; }}
        h1 {{ margin: 0 0 8px; font-size: clamp(2rem, 5vw, 3.4rem); line-height: 1; }}
        .lede {{ color: var(--muted); max-width: 70ch; }}
        .meta {{ color: var(--muted); font-size: 0.95rem; }}
        .metrics {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 14px; margin: 28px 0; }}
        .metric {{ background: var(--panel); border: 1px solid var(--line); border-radius: 16px; padding: 16px; box-shadow: 0 8px 24px rgba(68, 45, 24, 0.06); }}
        .metric h2 {{ margin: 0 0 8px; font-size: 0.95rem; color: var(--muted); }}
        .metric p {{ margin: 0; font-size: 2rem; color: var(--accent); }}
        section {{ margin-top: 28px; }}
        section h2 {{ margin: 0 0 12px; font-size: 1.3rem; }}
        .table-wrap {{ overflow-x: auto; background: var(--panel); border: 1px solid var(--line); border-radius: 16px; box-shadow: 0 8px 24px rgba(68, 45, 24, 0.06); }}
        table {{ width: 100%; border-collapse: collapse; min-width: 900px; }}
        th, td {{ padding: 10px 12px; text-align: left; border-bottom: 1px solid var(--line); vertical-align: top; }}
        th {{ position: sticky; top: 0; background: #f9f3ea; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 0.04em; color: var(--muted); }}
        tbody tr:nth-child(even) {{ background: rgba(166, 75, 42, 0.04); }}
        .files {{ display: flex; flex-wrap: wrap; gap: 10px; margin-top: 18px; }}
        .files a {{ color: var(--accent); text-decoration: none; background: var(--accent-soft); padding: 8px 10px; border-radius: 999px; }}
        .files a:hover {{ text-decoration: underline; }}
        @media (max-width: 640px) {{
            main {{ padding: 24px 14px 36px; }}
            table {{ min-width: 720px; }}
        }}
    </style>
</head>
<body>
    <main>
        <header>
            <p class=\"meta\">Generated {html.escape(generated_at)}</p>
            <h1>WAF Graph Analysis</h1>
            <p class=\"lede\">Local HTML report for the latest WAF analysis run. This report is regenerated from the current CSV outputs and highlights the strongest flagged IPs, the highest-scoring sessions, and the most anomalous clusters.</p>
            <div class=\"files\">
                <a href=\"cleaned_waf_events.csv\">cleaned_waf_events.csv</a>
                <a href=\"session_features.csv\">session_features.csv</a>
                <a href=\"session_similarity_edges.csv\">session_similarity_edges.csv</a>
                <a href=\"cluster_summary.csv\">cluster_summary.csv</a>
                <a href=\"ip_summary.csv\">ip_summary.csv</a>
            </div>
        </header>
        <section class=\"metrics\">{summary_html}</section>
        {render_html_table(top_ip_rows, ["ip", "candidate_score", "label", "sessions", "clusters", "high_confidence_sessions", "total_requests", "top_host", "top_webacl"], "Top Flagged IPs")}
        {render_html_table(top_session_rows, ["session_id", "ip", "cluster_id", "score", "requests", "hosts", "webacls", "suspicious_ratio", "block_ratio"], "Top Anomalous Sessions")}
        {render_html_table(top_cluster_rows, ["cluster_id", "label", "avg_score", "max_score", "sessions", "top_host", "top_webacl", "top_ip"], "Top Anomalous Clusters")}
    </main>
</body>
</html>
"""


def summarize_clusters(feature_rows: list[dict[str, object]]) -> list[dict[str, object]]:
    grouped: dict[int, list[dict[str, object]]] = defaultdict(list)
    for row in feature_rows:
        grouped[as_int(row["cluster_id"])].append(row)

    summaries: list[dict[str, object]] = []
    for cluster_id, rows in grouped.items():
        ranked_rows = sorted(rows, key=lambda row: (-as_float(row["anomaly_score"]), -as_int(row["requests"])))
        ips = Counter(as_str(row["ip"]) for row in rows if as_str(row["ip"]))
        hosts = Counter(as_str(row["top_host"]) for row in rows if as_str(row["top_host"]))
        webacls = Counter(as_str(row["top_webacl"]) for row in rows if as_str(row["top_webacl"]))
        ua_families = Counter(as_str(row["top_ua_family"]) for row in rows if as_str(row["top_ua_family"]))
        rules = Counter(as_str(row["top_rule"]) for row in rows if as_str(row["top_rule"]))
        avg_score = sum(as_float(row["anomaly_score"]) for row in rows) / len(rows)
        avg_requests = sum(as_int(row["requests"]) for row in rows) / len(rows)
        avg_suspicious_ratio = sum(as_float(row["suspicious_ratio"]) for row in rows) / len(rows)
        avg_block_ratio = sum(as_float(row["block_ratio"]) for row in rows) / len(rows)
        summaries.append(
            {
                "cluster_id": cluster_id,
                "session_count": len(rows),
                "avg_anomaly_score": round(avg_score, 2),
                "max_anomaly_score": round(max(as_float(row["anomaly_score"]) for row in rows), 2),
                "avg_requests": round(avg_requests, 2),
                "avg_suspicious_ratio": round(avg_suspicious_ratio, 4),
                "avg_block_ratio": round(avg_block_ratio, 4),
                "top_ip": ips.most_common(1)[0][0] if ips else "",
                "top_host": hosts.most_common(1)[0][0] if hosts else "",
                "top_webacl": webacls.most_common(1)[0][0] if webacls else "",
                "top_ua_family": ua_families.most_common(1)[0][0] if ua_families else "",
                "top_rule": rules.most_common(1)[0][0] if rules else "",
                "representative_session_id": as_str(ranked_rows[0]["session_id"]) if ranked_rows else "",
                "representative_ip": as_str(ranked_rows[0]["ip"]) if ranked_rows else "",
                "representative_score": round(as_float(ranked_rows[0]["anomaly_score"]), 2) if ranked_rows else 0.0,
                "cluster_label": (
                    "Highly suspicious coordinated probing"
                    if avg_score >= 85
                    else "Suspicious automation"
                    if avg_score >= 65
                    else "Mixed or lower-confidence activity"
                ),
            }
        )

    return sorted(summaries, key=lambda row: (-as_float(row["avg_anomaly_score"]), -as_int(row["session_count"])))


def summarize_ips(feature_rows: list[dict[str, object]], block_signal_weight: float = 10.0) -> list[dict[str, object]]:
    grouped: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in feature_rows:
        ip = as_str(row["ip"])
        if ip:
            grouped[ip].append(row)

    summaries: list[dict[str, object]] = []
    for ip, rows in grouped.items():
        ranked_rows = sorted(rows, key=lambda row: (-as_float(row["anomaly_score"]), -as_int(row["requests"])))
        hosts = Counter(as_str(row["top_host"]) for row in rows if as_str(row["top_host"]))
        webacls = Counter(as_str(row["top_webacl"]) for row in rows if as_str(row["top_webacl"]))
        ua_families = Counter(as_str(row["top_ua_family"]) for row in rows if as_str(row["top_ua_family"]))
        cluster_ids = {as_int(row["cluster_id"]) for row in rows}
        high_confidence_rows = [row for row in rows if as_float(row["anomaly_score"]) >= 85.0]
        avg_score = sum(as_float(row["anomaly_score"]) for row in rows) / len(rows)
        avg_block_ratio = sum(as_float(row["block_ratio"]) for row in rows) / len(rows)
        avg_suspicious_ratio = sum(as_float(row["suspicious_ratio"]) for row in rows) / len(rows)
        total_requests = sum(as_int(row["requests"]) for row in rows)
        candidate_score = min(
            avg_score * 0.50
            + min(len(high_confidence_rows) * 8.0, 25.0)
            + min(len(cluster_ids) * 2.0, 15.0)
            + min(avg_suspicious_ratio * 20.0, 10.0)
            + min(avg_block_ratio * block_signal_weight, 10.0),
            100.0,
        )
        summaries.append(
            {
                "ip": ip,
                "session_count": len(rows),
                "cluster_count": len(cluster_ids),
                "high_confidence_session_count": len(high_confidence_rows),
                "avg_anomaly_score": round(avg_score, 2),
                "max_anomaly_score": round(max(as_float(row["anomaly_score"]) for row in rows), 2),
                "avg_block_ratio": round(avg_block_ratio, 4),
                "avg_suspicious_ratio": round(avg_suspicious_ratio, 4),
                "total_requests": total_requests,
                "top_host": hosts.most_common(1)[0][0] if hosts else "",
                "top_webacl": webacls.most_common(1)[0][0] if webacls else "",
                "top_ua_family": ua_families.most_common(1)[0][0] if ua_families else "",
                "representative_session_id": as_str(ranked_rows[0]["session_id"]) if ranked_rows else "",
                "representative_session_score": round(as_float(ranked_rows[0]["anomaly_score"]), 2) if ranked_rows else 0.0,
                "candidate_ip_score": round(candidate_score, 2),
                "candidate_label": (
                    "Highly anomalous IP" if candidate_score >= 85.0 else "Flagged IP" if candidate_score >= 65.0 else "Lower-confidence IP"
                ),
            }
        )

    return sorted(
        summaries,
        key=lambda row: (
            -as_float(row["candidate_ip_score"]),
            -as_int(row["high_confidence_session_count"]),
            -as_int(row["session_count"]),
        ),
    )


def main() -> None:
    parser = argparse.ArgumentParser(description="Analyze WAF logs for anomalous session, cluster, and IP behavior.")
    parser.add_argument("csv_path", type=Path, help="Path to the raw WAF CSV file")
    parser.add_argument("--output-dir", type=Path, default=Path("output"), help="Directory for generated outputs")
    parser.add_argument("--gap-minutes", type=int, default=15, help="Session gap in minutes")
    parser.add_argument("--min-requests", type=int, default=25, help="Minimum requests required to keep a session")
    parser.add_argument("--similarity-threshold", type=float, default=0.60, help="Threshold for session similarity links")
    parser.add_argument("--partition-by", choices=["none", "host", "webacl"], default="host", help="Restrict graph links to the same host or WAF boundary")
    parser.add_argument(
        "--block-signal-weight",
        type=float,
        default=10.0,
        help="How much blocked-request ratio contributes to session and IP scoring; set to 0 to ignore it",
    )
    parser.add_argument("--seed-ips-file", type=Path, default=None, help="Optional newline-delimited file of known suspicious IPs")
    parser.add_argument("--seed-session-ids-file", type=Path, default=None, help="Optional newline-delimited file of known suspicious session IDs")
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)
    cleaned_csv_path = args.output_dir / "cleaned_waf_events.csv"
    cleaned_csv_path, cleaned_rows = prepare_clean_dataset(args.csv_path, cleaned_csv_path)

    seed_ips = load_seed_values(args.seed_ips_file)
    seed_session_ids = load_seed_values(args.seed_session_ids_file)

    sessions = build_sessions(cleaned_csv_path, args.gap_minutes, args.min_requests)
    feature_rows = [session.to_feature_row(args.block_signal_weight) for session in sessions]
    for row in feature_rows:
        seed_flag = as_str(row["ip"]) in seed_ips or as_str(row["session_id"]) in seed_session_ids
        row["seed_match"] = truthy_int(seed_flag)
        if seed_flag:
            row["anomaly_score"] = round(min(as_float(row["anomaly_score"]) + 15.0, 100.0), 2)

    edges, cluster_ids = build_similarity_graph(feature_rows, args.similarity_threshold, args.partition_by)
    for row in feature_rows:
        row["cluster_id"] = cluster_ids.get(as_str(row["session_id"]), 0)

    cluster_summaries = summarize_clusters(feature_rows)
    ip_summaries = summarize_ips(feature_rows, args.block_signal_weight)

    session_fieldnames = [
        "session_id",
        "cluster_id",
        "seed_match",
        "ip",
        "start_time",
        "end_time",
        "duration_seconds",
        "requests",
        "blocked_requests",
        "suspicious_requests",
        "requests_per_minute",
        "median_interarrival_seconds",
        "unique_hosts",
        "unique_webacls",
        "unique_uris",
        "unique_rules",
        "unique_ja4",
        "unique_ua_families",
        "block_ratio",
        "suspicious_ratio",
        "query_ratio",
        "cookie_ratio",
        "referer_ratio",
        "accept_language_ratio",
        "accept_encoding_ratio",
        "avg_header_count",
        "ua_churn",
        "top_host",
        "top_webacl",
        "top_rule",
        "top_ua_family",
        "anomaly_score",
    ]
    edge_fieldnames = [
        "source_session_id",
        "target_session_id",
        "similarity",
        "uri_similarity",
        "rule_similarity",
        "webacl_similarity",
        "ja4_similarity",
    ]
    cluster_fieldnames = [
        "cluster_id",
        "session_count",
        "avg_anomaly_score",
        "max_anomaly_score",
        "avg_requests",
        "avg_suspicious_ratio",
        "avg_block_ratio",
        "top_ip",
        "top_host",
        "top_webacl",
        "top_ua_family",
        "top_rule",
        "representative_session_id",
        "representative_ip",
        "representative_score",
        "cluster_label",
    ]
    ip_fieldnames = [
        "ip",
        "session_count",
        "cluster_count",
        "high_confidence_session_count",
        "avg_anomaly_score",
        "max_anomaly_score",
        "avg_block_ratio",
        "avg_suspicious_ratio",
        "total_requests",
        "top_host",
        "top_webacl",
        "top_ua_family",
        "representative_session_id",
        "representative_session_score",
        "candidate_ip_score",
        "candidate_label",
    ]

    write_csv(args.output_dir / "session_features.csv", feature_rows, session_fieldnames)
    write_csv(args.output_dir / "session_similarity_edges.csv", edges, edge_fieldnames)
    write_csv(args.output_dir / "cluster_summary.csv", cluster_summaries, cluster_fieldnames)
    write_csv(args.output_dir / "ip_summary.csv", ip_summaries, ip_fieldnames)
    write_text(
        args.output_dir / "analysis_visuals.html",
        build_html_report(cleaned_rows, feature_rows, edges, cluster_summaries, ip_summaries),
    )

    ranked_sessions = sorted(feature_rows, key=lambda row: (-as_float(row["anomaly_score"]), -as_int(row["requests"])))
    print(f"sessions={len(feature_rows)}")
    print(f"graph_edges={len(edges)}")
    print(f"clusters={len(cluster_summaries)}")
    print(f"flagged_ips={len(ip_summaries)}")
    print(f"cleaned_rows={cleaned_rows}")
    print(f"cleaned_csv={cleaned_csv_path}")
    print(f"partition_by={args.partition_by}")
    print(f"block_signal_weight={args.block_signal_weight}")
    print(f"seed_matches={sum(as_int(row['seed_match']) for row in feature_rows)}")
    print("top_flagged_ips=")
    for row in ip_summaries[:10]:
        print(
            row["ip"],
            f"candidate_score={row['candidate_ip_score']}",
            f"high_confidence_sessions={row['high_confidence_session_count']}",
            f"clusters={row['cluster_count']}",
            f"top_host={row['top_host']}",
        )
    print("top_anomalous_sessions=")
    for row in ranked_sessions[:10]:
        print(
            row["session_id"],
            row["ip"],
            f"cluster={row['cluster_id']}",
            f"score={row['anomaly_score']}",
            f"requests={row['requests']}",
            f"hosts={row['unique_hosts']}",
            f"webacls={row['unique_webacls']}",
            f"suspicious_ratio={row['suspicious_ratio']}",
            f"block_ratio={row['block_ratio']}",
        )


if __name__ == "__main__":
    main()